# VITAL-SOC | Live SOC Triage

Upload this notebook to Google Colab and select **Runtime → Run all**. The final cell starts the interactive webpage and displays its temporary link. In the dashboard, choose a scenario and press **Load + analyse sample**; it now loads and analyses the sample in one action. Keep the Colab runtime connected while presenting.

In [ ]:
# Install the app and start its public Colab webpage.
import base64, io, zipfile, pathlib, subprocess, sys, os, time, re, html
from IPython.display import display, HTML

APP_DIR = pathlib.Path('/content/vital_soc_analysis_fixed')
APP_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as archive:
    archive.extractall(APP_DIR)

print('Installing VITAL-SOC requirements...', flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '-r', str(APP_DIR / 'requirements.txt')])
print('Starting the interactive SOC dashboard...', flush=True)

# Stop a previous app started by this notebook, if any.
old = globals().get('_vital_soc_process')
if old is not None and old.poll() is None:
    old.terminate()
    try:
        old.wait(timeout=5)
    except Exception:
        old.kill()

log_path = APP_DIR / 'gradio_launch.log'
with open(log_path, 'w', encoding='utf-8') as log_handle:
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['GRADIO_ANALYTICS_ENABLED'] = 'False'
    _vital_soc_process = subprocess.Popen(
        [sys.executable, 'app.py', '--share', '--host', '0.0.0.0', '--port', '7860'],
        cwd=str(APP_DIR), stdout=log_handle, stderr=subprocess.STDOUT,
        env=env, start_new_session=True
    )

public_url = None
launch_log = ''
print('Waiting for the Colab website link...', flush=True)
for attempt in range(240):
    try:
        launch_log = log_path.read_text(encoding='utf-8', errors='replace')
    except Exception:
        launch_log = ''
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', launch_log)
    if match:
        public_url = match.group(0)
        break
    if _vital_soc_process.poll() is not None:
        break
    time.sleep(1)
    if attempt and attempt % 20 == 0:
        print(f'App is starting ({attempt} seconds)...', flush=True)

if public_url:
    display(HTML(
        '<div style="padding:16px;border-radius:12px;background:#eaf5ff;border:1px solid #9bc9e8">'
        '<div style="font-weight:700;margin-bottom:8px">VITAL-SOC is ready</div>'
        f'<a href="{html.escape(public_url)}" target="_blank" '
        'style="display:inline-block;padding:12px 18px;background:#145477;color:white;'
        'border-radius:8px;text-decoration:none;font-weight:700">OPEN VITAL-SOC WEBSITE</a>'
        '<div style="margin-top:8px;font-size:12px">Keep this Colab runtime connected while presenting.</div>'
        '</div>'
    ))
    print('Live website:', public_url)
else:
    print('The website link was not generated. Latest server output:')
    print(launch_log[-10000:] if launch_log else 'No server output was produced.')
    print('Check Colab internet access, then rerun this cell.')
